# OmniSpeak (Colab)

Đọc văn bản, nhân bản giọng nói, lưu thư viện giọng — chạy trên Google Colab, dùng model [OmniVoice](https://github.com/k2-fsa/OmniVoice) (`k2-fsa/OmniVoice`, Apache-2.0).

Code app (frontend + backend) nằm trên GitHub (`trkhanh8312-make/omnispeak`), notebook chỉ `git clone`/`git pull` về rồi chạy — không nhúng code app trong notebook.

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy các cell theo thứ tự từ trên xuống — mọi cell đều an toàn khi chạy lại.

## Cài đặt

In [ ]:
# 1. GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Không có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")

In [ ]:
# 2. Cài đặt
import subprocess
import sys

def run(cmd, what=""):
    print("\n$", cmd if isinstance(cmd, str) else " ".join(cmd))
    if subprocess.run(cmd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"Lỗi: {what or cmd}. Xem log phía trên rồi chạy lại cell này.")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1")
run([sys.executable, "-m", "pip", "install", "-q",
     "omnivoice", "fastapi", "uvicorn[standard]", "python-multipart", "soundfile"])
run([sys.executable, "-c",
     "from omnivoice import OmniVoice, VoiceClonePrompt; import fastapi, soundfile; print('OK')"])

In [ ]:
# 3. Lấy code (frontend + backend) từ GitHub — luôn đồng bộ bản mới nhất
import os

REPO_DIR = "/content/omnispeak_repo"
GITHUB_USER = "trkhanh8312-make"
GITHUB_REPO = "omnispeak"
GITHUB_BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(f"git -C {REPO_DIR} fetch origin {GITHUB_BRANCH}", "git fetch")
    run(f"git -C {REPO_DIR} reset --hard origin/{GITHUB_BRANCH}", "git reset --hard")
else:
    run(f"git clone --branch {GITHUB_BRANCH} {REPO_URL} {REPO_DIR}", "git clone")

FRONTEND_DIR = os.path.join(REPO_DIR, "frontend")
APP_DIR = os.path.join(REPO_DIR, "backend")

for required in (os.path.join(FRONTEND_DIR, "index.html"), os.path.join(APP_DIR, "backend.py")):
    if not os.path.isfile(required):
        raise SystemExit(
            f"Không thấy {required}.\n"
            "Kiểm tra lại: file đã push lên GitHub chưa, và đường dẫn trong repo "
            "(frontend/index.html, backend/backend.py) có đúng không."
        )

print(f"Đã đồng bộ code từ {REPO_URL} (nhánh {GITHUB_BRANCH}).")

In [ ]:
# 4. Mount Google Drive (tuỳ chọn) — lưu bền vững giọng nói + model đã tải
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/omnispeak_data"
HF_DIR = "/content/drive/MyDrive/omnispeak_hf_cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
os.environ["OMNISPEAK_DATA_DIR"] = DATA_DIR
os.environ["HF_HOME"] = HF_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # Drive (FUSE) không hỗ trợ symlink
print("Giọng nói + model sẽ lưu bền vững trên Drive.")

In [ ]:
# 5. Tải trước model (bỏ qua cũng được — sẽ tự tải khi khởi động backend)
from huggingface_hub import snapshot_download

print("Model tại:", snapshot_download("k2-fsa/OmniVoice"))

In [ ]:
# 6. Khởi động backend
FORCE_RESTART = True  # luôn nạp lại code mới nhất đã đồng bộ ở cell 3

import json
import os
import subprocess
import sys
import time
import urllib.request

PORT = 3900
LOG_PATH = "/content/omnispeak_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = None if FORCE_RESTART else health()
if info:
    print("Backend đang chạy —", info)
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNISPEAK_DATA_DIR"] = os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data")
    env["OMNISPEAK_FRONTEND_DIR"] = FRONTEND_DIR
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend:app", "--app-dir", APP_DIR,
         "--host", "127.0.0.1", "--port", str(PORT)],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Đang khởi động (PID {proc.pid})...")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print("Backend đã sẵn sàng —", info)
    else:
        try:
            tail = "".join(open(LOG_PATH, errors="replace").readlines()[-40:])
        except OSError:
            tail = "(không có log)"
        raise SystemExit(f"Backend không lên được sau 5 phút.\n--- log ---\n{tail}")

In [ ]:
# 7. Mở giao diện web
from google.colab import output

output.serve_kernel_port_as_window(3900)

### Tổng kết

Danh sách giọng nói đã lưu.

In [ ]:
# Tổng kết
import requests

try:
    profiles = requests.get("http://127.0.0.1:3900/profiles", timeout=15).json()
    print(f"Giọng đã lưu ({len(profiles)}):")
    for p in profiles:
        print(" ", p["id"], p.get("name"))
except Exception as e:
    print("Không lấy được danh sách:", e)

## Xử lý sự cố

- **`device: cpu` hoặc generate chậm** — bật GPU: Runtime → Change runtime type → T4 GPU.
- **Sửa code (`frontend/index.html` hoặc `backend/backend.py`) xong mà chạy vẫn y nguyên** — push code mới lên GitHub, rồi chạy lại cell 3 (đồng bộ code) và cell 6 (`FORCE_RESTART = True` tự nạp lại, không cần tự kill process).
- **Muốn giữ giọng nói + model qua các phiên sau** — chạy cell 4 (mount Drive) trước cell 5.
- **Tab UI trắng hoặc lỗi** — chạy lại cell 6 rồi cell 7. Cho phép pop-up cho `colab.research.google.com`; hoặc đổi cell 7 sang `output.serve_kernel_port_as_iframe(3900)` để nhúng UI ngay trong notebook.
- **Không đồng bộ được code ở cell 3** — kiểm tra repo có để public không (git clone qua HTTPS không đọc được repo private nếu chưa cấu hình token), và tên nhánh `GITHUB_BRANCH` có đúng không.
- **Xem log backend** — `/content/omnispeak_backend.log`.